In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import Dict, Tuple, List
from datahub.ingestion.graph.client import DataHubGraph, DatahubClientConfig
import dotenv
import pathlib
import json
import csv
import time

dotenv.load_dotenv()

from datahub_integrations.gen_ai.description_v2 import (
    generate_column_descriptions_for_urn,
    parse_llm_output,
)

In [ ]:
import cachetools
from datahub_integrations.gen_ai.cached_graph import make_cached_graph


current_dir = pathlib.Path().resolve()
graph_credentials = json.loads(
    (
        current_dir / "experiments" / "docs_generation" / "graph_credentials.json"
    ).read_text()
)
assert "acryl" in graph_credentials


@cachetools.cached(cache=cachetools.Cache(maxsize=1000))
def create_datahub_graph(key: str) -> DataHubGraph:
    """Create a DataHub client based on the graph credentials."""

    creds = graph_credentials[key]
    graph = DataHubGraph(
        DatahubClientConfig(
            **creds,
        )
    )
    graph.test_connection()

    cached_graph = make_cached_graph(graph, ttl=0)
    cached_graph.test_connection()
    return cached_graph


assert create_datahub_graph("acryl")
_test_entity_1 = create_datahub_graph("longtailcompanions").get_entity_semityped(
    "urn:li:dataset:(urn:li:dataPlatform:snowflake,long_tail_companions.adoption.pet_profiles,PROD)"
)
assert isinstance(_test_entity_1, dict)
_test_entity_2 = create_datahub_graph("longtailcompanions").get_entity_semityped(
    "urn:li:dataset:(urn:li:dataPlatform:snowflake,datahub_community.datahub_slack.message,PROD)"
)
assert isinstance(_test_entity_2, dict)
assert _test_entity_1["datasetKey"] != _test_entity_2["datasetKey"]  # type: ignore
del _test_entity_1, _test_entity_2

In [ ]:
# Test the bedrock credentials.
from datahub_integrations.gen_ai.description_v2 import call_bedrock_llm


print(call_bedrock_llm("What is the capital of France?", max_tokens=100))

In [ ]:
def write_llm_output_to_csv(llm_response, csv_path) -> None:
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        csvwriter = csv.writer(csvfile)

        csvwriter.writerow(["instance", "urn", "column_description"])
        for instance, urn, value in llm_response:
            csvwriter.writerow([instance, urn, value])
    print("csv file created successfully!!!")


def generate_column_descriptions(
    urns_dict: Dict[str, List[str]]
) -> Tuple[List[Tuple[str, str, str]], dict]:
    """
    This function also returns column_infos for debugging purpose (To check the if metadata information is generated correctly) and can be removed
    """
    llm_responses = []
    column_infos = {}
    # graph_client = create_datahub_graph("chime")
    for instance, urns in urns_dict.items():
        graph_client = create_datahub_graph(instance)
        for urn in urns:
            print("DB: ", instance, "URN: ", urn)
            column_descriptions, column_info = generate_column_descriptions_for_urn(
                graph_client, urn
            )
            column_infos[urn] = column_info
            llm_responses.append((instance, urn, column_descriptions))
    return llm_responses, column_infos


def get_parsed_responses(
    llm_responses: List[Tuple[str, str, str]]
) -> List[Tuple[str, str, Dict[str, str]]]:
    parsed_llm_responses = []
    for instance, urn, column_descriptions in llm_responses:
        if column_descriptions is None:
            extracted_dict = None
        else:
            extracted_dict = parse_llm_output(column_descriptions)
        parsed_llm_responses.append((instance, urn, extracted_dict))
    return parsed_llm_responses

In [ ]:
urns_dict: Dict[str, List[str]] = json.loads(
    (current_dir / "experiments" / "docs_generation" / "test_urns.json").read_text()
)

exec_time = time.time()
PARSED_OUTPUT_CSV_PATH = f"parsed_llm_output_{exec_time}.csv"
OUTPUT_CSV_PATH = f"llm_output_{exec_time}.csv"

llm_responses, column_infos = generate_column_descriptions(
    urns_dict, model_config={"max_tokens": 5000}
)
write_llm_output_to_csv(llm_responses, OUTPUT_CSV_PATH)
parsed_llm_responses = get_parsed_responses(llm_responses)
write_llm_output_to_csv(parsed_llm_responses, PARSED_OUTPUT_CSV_PATH)

In [ ]:
llm_responses

In [ ]:
text = llm_responses[-2][2]
print(text)
# parse_llm_output(text)